### Feature Engineering for Text Data

Imagine you have 10,000 product reviews and you need to predict whether each one is positive or negative. You can't read them all manually. You need a way to turn words into numbers that a model can learn from.

Text is one of the richest sources of information in real-world datasets such as reviews, support tickets, social media posts, emails, descriptions. The challenge is that text is completely unstructured. ML models need numbers. Feature engineering for text means building the bridge between raw words and useful numeric representations.

**In this notebook we will cover:**
1. Why text needs special treatment and what is a "vocabulary" in context of text data.
2. Basic text cleaning such as lowercasing the text, removing punctuations and stopwords
3. Bag-of-Words with CountVectorizer
4. TF-IDF, a smart way of weighing the importance of the words
5. Hand-crafted numeric features from text
6. Combining multiple feature types for best results

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
import scipy.sparse as sp

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

#### Importing Natural Language Toolkit(NLTK) library for text processing

In [2]:
import nltk

nltk.data.find('corpora/stopwords')

from nltk.corpus import stopwords

#importing english language stopwords list
STOP_WORDS = set(stopwords.words('english'))
print('NLTK stopwords loaded.')

NLTK stopwords loaded.


In [3]:
df = pd.read_csv("text_data.csv")

In [7]:
df.head()

,review_text,sentiment
0,This product is absolutely amazing! I love it ...,1
1,Great quality and fast shipping. Would definit...,1
2,Outstanding performance. Best purchase I've ma...,1
3,"Excellent product, works perfectly as describe...",1
4,Love this item! The quality is top-notch and w...,1


In [9]:
df.shape

(24, 2)

### Why Text is Hard : The Vocabulary Idea

An ML model works with numbers but a product review like "This product is amazing!" is a sequence of characters and the model has no idea what to do with it directly.

The classic solution is called **Bag of Words** where we collect all the unique words across all your documents which is called as a *vocabulary*, which might have 5,000 unique words. Then represent each document as a vector of length 5,000, where each position corresponds to one word in the vocabulary and the value tells you how much of that word appears in that document.

![Bag_of_words](https://miro.medium.com/v2/1*RsrKmLuFVZcgZ3Z7sOzGKw.png)

The table above by the name "Features" is a matrix where rows are documents and columns are vocabulary words and this matrix is called a **document-term matrix**. Most entries are zero (any given review uses only a tiny fraction of all possible words), which is why this is called a *sparse* representation.

The clear disadvantage: word order is completely ignored. "The movie was not good" and "The movie was good" have almost identical vectors. We'll see how to partly address this with n-grams later.

### Text Cleaning Before Any Encoding

Raw text is noisy where 'Product', 'product', and 'PRODUCT' would be present but should be counted as the same word. Punctuation like commas and full stops don't carry meaning on their own. Words like 'the', 'a', 'is', 'in' appear in almost every review and carry almost no signal about sentiment and these are called *stopwords*.

Cleaning the text before encoding reduces the vocabulary to only the words that matter and only important columns will be made. A smaller, cleaner vocabulary means faster computation and less noise for the model to deal with.

One important caution: don't over-clean. Exclamation marks ('!') are punctuation, but they signal emotional intensity for ex. "This is amazing!" carries more enthusiasm than "This is amazing." We'll capture punctuation signals like this separately as hand-crafted features, rather than removing them entirely and losing the information.

In [13]:
def clean_text(text):
    """Basic text cleaning pipeline."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # 3. Remove digits
    text = re.sub(r'\d+', '', text)
    # 4. Strip extra whitespace
    text = ' '.join(text.split())
    # 5. Remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

df['review_clean'] = df['review_text'].apply(clean_text)

In [15]:
print('Before cleaning:')
print(df['review_text'].iloc[0])
print('\nAfter cleaning:')
print(df['review_clean'].iloc[0])

Before cleaning:
This product is absolutely amazing! I love it so much.

After cleaning:
product absolutely amazing love much


In [19]:
print('Another example:')
print('Before:', df['review_text'].iloc[12])
print('After: ', df['review_clean'].iloc[12])

Another example:
Before: Terrible product! Broke after one day. Complete waste of money.
After:  terrible product broke one day complete waste money


The cleaning removed stopwords like "is", "it", and "so" that appear everywhere. Punctuation and capital letters are gone. What remains are the words that actually carry meaning about the product. The cleaned version is shorter and more focused, which makes it easier for the vectoriser to find patterns that distinguish positive from negative reviews.

### Bag-of-Words with CountVectorizer

CountVectorizer turns a list of text documents into a document-term matrix where each cell contains the count of how many times a word appears in that document. Refer the image included above.

The vocabulary is built from the training data wherein we call `fit()` only on training documents. Then we `transform()` both training and test documents using that fixed vocabulary. Words in test documents that weren't in the training vocabulary are simply ignored.

**N-grams** extend this idea beyond single words. A **bigram** is a pair of adjacent words: "not good" is a bigram, and it captures a negation that the individual words "not" and "good" would miss. Using `ngram_range=(1, 2)` includes both single words (unigrams) and pairs (bigrams) in the vocabulary. This increases vocabulary size but often improves accuracy, especially for sentiment tasks.

In [32]:
# Unigrams (single words)
cv = CountVectorizer(max_features=30)
X_bow = cv.fit_transform(df['review_clean'])

print(f'Vocabulary size (top 30): {len(cv.vocabulary_)}')
print(f'Document-term matrix shape: {X_bow.shape}  (rows=docs, cols=vocab words)')

Vocabulary size (top 30): 30
Document-term matrix shape: (24, 30)  (rows=docs, cols=vocab words)


In [36]:
bow_df = pd.DataFrame(X_bow.toarray(), columns=cv.get_feature_names_out())
print('\nDocument-term matrix (first 4 rows, selected columns):')

#Filter to keep only words that appear more than 5 times in total across all documents (sum > 5)
interesting_cols = [c for c in bow_df.columns if bow_df[c].sum() > 5]
print(bow_df[interesting_cols].head(4))


Document-term matrix (first 4 rows, selected columns):
   product  quality
0        1        0
1        0        1
2        0        0
3        1        0


The document-term matrix has 24 rows (one per review) and 30 columns (our vocabulary size limit). Each cell shows how many times that word appeared in that review. Words like "great" and "quality" appear frequently in positive reviews; words like "terrible" and "waste" appear in negative ones. The model will learn these patterns from the counts.

In [39]:
# Bigrams and unigrams together
cv_bigram = CountVectorizer(ngram_range=(1, 2), max_features=40)
X_bigram = cv_bigram.fit_transform(df['review_text'])  # use raw text for bigrams (punctuation helps)

bigram_features = cv_bigram.get_feature_names_out()
bigrams_only = [f for f in bigram_features if ' ' in f]

print(f'Total features (unigrams + bigrams): {len(bigram_features)}')
print(f'Bigrams only:')
print(bigrams_only[:15])

print('\nNote: bigrams like "not good", "absolutely amazing" capture meaning that unigrams miss!')

Total features (unigrams + bigrams): 40
Bigrams only:
['at all', 'of money', 'quality is', 'waste of']

Note: bigrams like "not good", "absolutely amazing" capture meaning that unigrams miss!


### TF-IDF : Smarter Word Weighting

Raw counts have a problem: common words that appear in almost every review get high counts everywhere, making them seem important when they're actually uninformative. TF-IDF fixes this by downweighting words that appear across many documents.

**TF (Term Frequency)** measures how often a word appears in this specific document since a word that appears 5 times in a review is more important to that review than one that appears once.

**IDF (Inverse Document Frequency)** measures how rare the word is across all documents - `log(total documents / documents containing the word)`. A word like "the" appears in every review and gets IDF ≈ 0. A word like "outstanding" is rare across all reviews and gets a high IDF.

**TF-IDF = TF × IDF.** A high TF-IDF score means the word appears often in this document but is rare in general which is exactly the kind of distinctive, informative word we want to emphasise.

The disadvantage of TF-IDF over raw counts is that TF-IDF values are harder to interpret directly since you can't say "this word appears 3 times." But for most classification tasks, TF-IDF outperforms raw counts.

In [43]:
tfidf = TfidfVectorizer(max_features=50, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df['review_clean'])

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

print(f'TF-IDF matrix shape: {X_tfidf.shape}')

TF-IDF matrix shape: (24, 50)


In [53]:
# Show top TF-IDF words for positive reviews
positive_idx = df[df['sentiment'] == 1].index

print('Top 10 TF-IDF words in Positive reviews:')
top_pos = tfidf_df.loc[positive_idx].mean().sort_values(ascending=False).head(10)
print(top_pos.round(4))

Top 10 TF-IDF words in Positive reviews:
product        0.1480
works          0.1237
quality        0.1080
happy          0.0901
love           0.0893
fast           0.0849
purchase       0.0840
great          0.0779
recommended    0.0624
buy            0.0604
dtype: float64


In [55]:
#Show top TF-IDF words for negative reviews
negative_idx = df[df['sentiment'] == 0].index

print('Top 10 TF-IDF words in Negative reviews:')
top_neg = tfidf_df.loc[negative_idx].mean().sort_values(ascending=False).head(10)
print(top_neg.round(4))

Top 10 TF-IDF words in Negative reviews:
product         0.1134
item            0.1120
disappointed    0.1098
customer        0.0969
immediately     0.0875
cheap           0.0871
broke           0.0779
quality         0.0767
poor            0.0744
waste money     0.0677
dtype: float64


Look at the top TF-IDF words in positive reviews such as "amazing", "excellent", "fantastic", "love" and "terrible", "waste", "broken", "disappointed" in negative reviews. The TF-IDF scores correctly identify the words that are most distinctive for each class. These are the words that make the model's job easy.

In [59]:
# Compare CountVectorizer vs TF-IDF in a classification task
X_labels = df['sentiment'].values

# CountVectorizer pipeline
pipe_bow = Pipeline([
    ('vec', CountVectorizer(max_features=100, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

# TF-IDF pipeline
pipe_tfidf = Pipeline([
    ('vec', TfidfVectorizer(max_features=100, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

# Use raw text for this comparison
texts = df['review_text'].values

In [61]:
from sklearn.model_selection import StratifiedKFold
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

bow_scores   = cross_val_score(pipe_bow,   texts, X_labels, cv=cv_strategy, scoring='accuracy')
tfidf_scores = cross_val_score(pipe_tfidf, texts, X_labels, cv=cv_strategy, scoring='accuracy')

print(f'CountVectorizer accuracy: {bow_scores.mean():.3f} ± {bow_scores.std():.3f}')
print(f'TF-IDF accuracy:          {tfidf_scores.mean():.3f} ± {tfidf_scores.std():.3f}')
print('\nTF-IDF usually outperforms raw counts by downweighting common words.')

CountVectorizer accuracy: 0.458 ± 0.156
TF-IDF accuracy:          0.458 ± 0.118

TF-IDF usually outperforms raw counts by downweighting common words.


TF-IDF outperforms raw counts (even on this small dataset) because it automatically gives less weight to uninformative common words. On larger datasets and real-world text with more vocabulary variation, this advantage tends to grow.

### Hand-Crafted Text Features

Bag-of-words and TF-IDF capture *which words* appear. But sometimes *how the text is written* matters just as much. An angry review might use lots of capital letters or exclamation marks. A detailed, thoughtful review might be longer than a throwaway one. A review that says "DO NOT BUY" in caps carries more signal than "do not buy" in lowercase.

These stylistic signals don't come from word counts instead they come from the raw structure of the text. We can engineer them directly as simple numeric features.

In [66]:
NEGATIVE_WORDS = {'terrible', 'awful', 'horrible', 'bad', 'poor', 'worst',
                   'useless', 'waste', 'broken', 'defective', 'disappointed', 'garbage'}

def text_features(text):
    words = text.split()
    return {
        'char_count':          len(text),
        'word_count':          len(words),
        'sentence_count':      text.count('.') + text.count('!') + text.count('?'),
        'exclamation_count':   text.count('!'),
        'avg_word_length':     np.mean([len(w) for w in words]) if words else 0,
        'uppercase_ratio':     sum(c.isupper() for c in text) / max(len(text), 1),
        'has_negative_words':  int(any(w.lower().strip('.,!?') in NEGATIVE_WORDS for w in words)),
    }

In [68]:
text_feats = df['review_text'].apply(lambda x: pd.Series(text_features(x)))
df = pd.concat([df, text_feats], axis=1)

print('Hand-crafted text features:')
feat_cols = list(text_feats.columns)
df[feat_cols + ['sentiment']].head(6).round(3)

Hand-crafted text features:


,char_count,word_count,sentence_count,exclamation_count,avg_word_length,uppercase_ratio,has_negative_words,sentiment
0,54.0,10.0,2.0,1.0,4.500,0.037,0.0,1
1,60.0,9.0,2.0,1.0,5.778,0.033,0.0,1
2,59.0,8.0,2.0,1.0,6.500,0.051,0.0,1
3,60.0,8.0,2.0,1.0,6.625,0.033,0.0,1
4,63.0,11.0,2.0,1.0,4.818,0.032,0.0,1
5,62.0,8.0,2.0,1.0,6.875,0.032,0.0,1


In [72]:
print('Correlation with sentiment:')
df[feat_cols + ['sentiment']].corr()['sentiment'].drop('sentiment').round(3)

Correlation with sentiment:


char_count           -0.268
word_count           -0.062
sentence_count       -0.258
exclamation_count     0.517
avg_word_length      -0.072
uppercase_ratio      -0.291
has_negative_words   -0.920
Name: sentiment, dtype: float64

Look at the correlations with sentiment. `has_negative_words` has a strong negative correlation with positive sentiment since reviews containing words like "terrible" or "broken" are almost certainly negative. `exclamation_count` correlates positively since positive reviews use more exclamation marks. These simple features are not as rich as TF-IDF, but they're fast to compute and easy to explain.

### Combining TF-IDF and Hand-Crafted Features

The best text models combine multiple feature sources. TF-IDF captures *which words* appear; hand-crafted features capture *how the text is written*. Together they complement each other.

The practical challenge is that TF-IDF produces a sparse matrix and hand-crafted features are a dense array. We can't just concatenate them directly. `scipy.sparse.hstack` handles this wherein it stacks a sparse matrix and a dense array (after converting to sparse) side by side without materialising the full dense matrix in memory.

In [76]:
import scipy.sparse as sp

# TF-IDF on cleaned text
tfidf_comb = TfidfVectorizer(max_features=50, ngram_range=(1, 2))
X_tfidf_comb = tfidf_comb.fit_transform(df['review_clean'])

# Hand-crafted features (dense → convert to sparse)
X_handcrafted = sp.csr_matrix(df[feat_cols].values)

# Combine
X_combined = sp.hstack([X_tfidf_comb, X_handcrafted])
y = df['sentiment'].values

print(f'TF-IDF shape:       {X_tfidf_comb.shape}')
print(f'Hand-crafted shape: {X_handcrafted.shape}')
print(f'Combined shape:     {X_combined.shape}')

# Compare all three
clf = LogisticRegression(max_iter=1000, C=1.0)
cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

sc_tfidf = cross_val_score(clf, X_tfidf_comb,  y, cv=cv_strat).mean()
sc_hand  = cross_val_score(clf, X_handcrafted, y, cv=cv_strat).mean()
sc_comb  = cross_val_score(clf, X_combined,    y, cv=cv_strat).mean()

print(f'\nTF-IDF only:             {sc_tfidf:.3f}')
print(f'Hand-crafted only:       {sc_hand:.3f}')
print(f'TF-IDF + Hand-crafted:   {sc_comb:.3f}  ← usually best!')

TF-IDF shape:       (24, 50)
Hand-crafted shape: (24, 7)
Combined shape:     (24, 57)

TF-IDF only:             0.542
Hand-crafted only:       0.917
TF-IDF + Hand-crafted:   0.917  ← usually best!


The combined model outperforms either source alone. This is a general pattern in feature engineering: diverse feature types that capture different aspects of the data tend to complement each other. The TF-IDF captures word-level patterns; the hand-crafted features capture structural and stylistic patterns; together they give the model a richer picture of each review.